# NSU BDA. 2024 Accidents
Соревнование для студенттов курса АБМД, ФИТ НГУ 2024

Студент: Митюшин Владимир 24221

In [503]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Input, BatchNormalization

Загрузим данные

In [504]:
X_train = pd.read_csv('../data/X_train.csv', index_col=0)
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)
Y_train = pd.read_csv('../data/Y_train.csv', index_col=0)

Выведем информацию по датасетам

## Очистка и подготовка данных

In [505]:
df = X_train.copy()

In [506]:
drop_indexes = df.index[df['vehicle_reference'] > 20].tolist()
drop_indexes= drop_indexes + df.index[df['age_of_vehicle'] > 25].tolist()
df.drop(drop_indexes, inplace=True)
Y_train.drop(drop_indexes, inplace=True)

df['sex_of_casualty'] = df['sex_of_casualty'].replace(9, -1)
df['car_passenger'] = df['car_passenger'].replace(9, -1)
df['casualty_type'] = df['casualty_type'].replace([3, 4, 5, 22, 23, 97, 103, 104, 105, 106], 2).replace([20, 21, 98, 113], 19)
df.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_home_area_type,casualty_distance_banding,vehicle_type,vehicle_manoeuvre,vehicle_left_hand_drive,engine_capacity_cc,age_of_vehicle,police_force,first_road_class,first_road_number,road_type,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.00000,496651.000000
mean,1.449964,1.473234,1.367147,36.783472,6.307860,0.760866,0.615100,0.220797,0.050619,0.026020,7.069497,1.066266,1.585230,9.259158,19.415094,1.279317,1233.865072,6.038945,28.314019,4.144252,800.691008,5.232447,37.519067,3.786178,1.679829,2.996621,223.642938,0.323237,1.107128,2.057342,1.63932,1.372052
std,0.593538,0.725776,0.527032,19.515047,2.453348,2.147334,1.965768,0.553609,0.436992,0.232706,9.218434,0.918710,1.423874,11.371726,21.913165,1.464006,1295.173901,6.155454,24.407437,1.470665,1593.561288,1.673314,14.734020,12.024648,2.516757,2.757975,936.358471,1.640897,2.375478,1.735276,1.79187,0.931780
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.000000,1.000000,0.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.00000,-1.000000
25%,1.000000,1.000000,1.000000,22.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,8.000000,9.000000,1.000000,108.000000,0.000000,5.000000,3.000000,0.000000,6.000000,30.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.00000,1.000000
50%,1.000000,1.000000,1.000000,34.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,1.000000,9.000000,18.000000,1.000000,1299.000000,5.000000,23.000000,4.000000,38.000000,6.000000,30.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.000000
75%,2.000000,2.000000,2.000000,50.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,2.000000,9.000000,18.000000,1.000000,1796.000000,11.000000,45.000000,6.000000,562.000000,6.000000,50.000000,3.000000,4.000000,6.000000,0.000000,0.000000,0.000000,4.000000,1.00000,2.000000
max,20.000000,3.000000,2.000000,102.000000,11.000000,10.000000,9.000000,2.000000,9.000000,2.000000,99.000000,3.000000,5.000000,99.000000,99.000000,9.000000,29980.000000,25.000000,99.000000,6.000000,9176.000000,9.000000,70.000000,99.000000,9.000000,6.000000,9999.000000,9.000000,9.000000,7.000000,9.00000,9.000000


In [507]:
df.drop(columns=['accident_index', 'vehicle_reference'], inplace=True)
df = df.drop(columns=['age_of_casualty', 'vehicle_left_hand_drive', 'pedestrian_movement', 'first_road_number', 
                      'second_road_number', 'junction_control', 'pedestrian_crossing_human_control'])

### Посмотрим на категориальные признаки

In [508]:
# df['generic_make_model'] = df['generic_make_model'].str.split(n=1).str[0].str.upper()
# display(df['generic_make_model'].value_counts(dropna=False))

df = df.drop(columns='generic_make_model')

In [509]:
df['local_authority_highway'] = df['local_authority_highway'].str[:4]
display(df['local_authority_highway'].value_counts(dropna=False))

# df = df.drop(columns='local_authority_highway')

local_authority_highway
E100    158518
E060    115077
E090    100761
E080     83493
S120     21799
W060     16905
EHEA        98
Name: count, dtype: int64

In [510]:
one_hot = pd.get_dummies(df.select_dtypes('O'), prefix=df.select_dtypes('O').columns, dtype=bool)
df = pd.concat([one_hot, df.select_dtypes('number'), df.select_dtypes('bool')], axis=1)

df.describe()

,casualty_class,sex_of_casualty,age_band_of_casualty,pedestrian_location,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_home_area_type,casualty_distance_banding,vehicle_type,vehicle_manoeuvre,engine_capacity_cc,age_of_vehicle,police_force,first_road_class,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.000000,496651.00000,496651.000000
mean,1.473234,1.367147,6.307860,0.760866,0.220797,0.050619,0.026020,7.069497,1.066266,1.585230,9.259158,19.415094,1233.865072,6.038945,28.314019,4.144252,5.232447,37.519067,3.786178,2.996621,1.107128,2.057342,1.63932,1.372052
std,0.725776,0.527032,2.453348,2.147334,0.553609,0.436992,0.232706,9.218434,0.918710,1.423874,11.371726,21.913165,1295.173901,6.155454,24.407437,1.470665,1.673314,14.734020,12.024648,2.757975,2.375478,1.735276,1.79187,0.931780
min,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.00000,-1.000000
25%,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,8.000000,9.000000,108.000000,0.000000,5.000000,3.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.000000
50%,1.000000,1.000000,6.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,1.000000,9.000000,18.000000,1299.000000,5.000000,23.000000,4.000000,6.000000,30.000000,1.000000,3.000000,0.000000,1.000000,1.00000,1.000000
75%,2.000000,2.000000,8.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,2.000000,9.000000,18.000000,1796.000000,11.000000,45.000000,6.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.00000,2.000000
max,3.000000,2.000000,11.000000,10.000000,2.000000,9.000000,2.000000,99.000000,3.000000,5.000000,99.000000,99.000000,29980.000000,25.000000,99.000000,6.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.00000,9.000000


## Обучение

### Подготовка данных

In [511]:
df_all = df.copy()
df_all['y'] = Y_train['casualty_severity']
df_2 = df_all[df_all['y'] != 1]

df_all = df_all.drop(columns='y')
df_2 = df_2.drop(columns='y')

Y_all = Y_train.copy()
Y_2 = Y_all.drop(index=Y_all.index[Y_all['casualty_severity'] == 1])
Y_1 = Y_all['casualty_severity'].replace(3, 2)
Y_2 = Y_2.apply(lambda y: y-2)
Y_1 = Y_1.apply(lambda y: y-1)

print(df_2.shape, Y_2.shape)
print(df_all.shape, Y_1.shape)
print(np.unique(Y_1), np.unique(Y_2))

(490725, 31) (490725, 1)
(496651, 31) (496651,)
[0 1] [0 1]


In [ ]:
scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(df_2, Y_2, random_state=42) # Y_train
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = keras.utils.to_categorical(y_train, num_classes=2) # _resampled
y_test = keras.utils.to_categorical(y_test, num_classes=2)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

(368043, 31) (368043, 1)
y_train shape: (368043, 2)
y_test shape: (122682, 2)


In [518]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(df_all, Y_1, random_state=42)
X_train1 = scaler.fit_transform(X_train1)
X_test1 = scaler.transform(X_test1)

y_train1 = keras.utils.to_categorical(y_train1, num_classes=2)
y_test1 = keras.utils.to_categorical(y_test1, num_classes=2)
print('y_train shape:', y_train1.shape)
print('y_test shape:', y_test1.shape)
print(X_train1.shape, y_train1.shape)

y_train shape: (372488, 2)
y_test shape: (124163, 2)
(372488, 31) (372488, 2)


### Подготовка модели

In [515]:
from keras.api.optimizers import Adam

def create_model(shape, seed):
    keras.utils.set_random_seed(seed)
    model = Sequential() 
    model.add(Input(shape=(shape, 1)))
    model.add(Flatten())
    model.add(BatchNormalization())
    
    model.add(Dense(240, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'  500
    model.add(Dense(140, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'  220
    model.add(Dropout(.5))
    model.add(Dense(140, activation='elu')) # 'relu', 'leaky_relu', 'elu'
    model.add(Dense(90, activation='elu'))
    model.add(Dropout(.25))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(2, activation='softmax')) # 'sigmoid', 'softmax', 'tanh'
    
    model.compile(optimizer=Adam(), 
                  loss='categorical_crossentropy', metrics=['f1_score'])
    return model

In [516]:
# from keras.api.callbacks import EarlyStopping
# from scikeras.wrappers import KerasClassifier
# from sklearn.model_selection import cross_val_score
# from sklearn.model_selection import KFold
# from bayes_opt import BayesianOptimization
# import warnings
# warnings.filterwarnings('ignore')

# def nn_cl_bo2(class1, class2):
#     class_weights = [{0:class1, 1:class2}, {0:class2, 1:class1}]
    
#     es = EarlyStopping(monitor='val_loss', mode='min', verbose=0, patience=10)
#     nn = KerasClassifier(model=create_model(142), epochs=70, batch_size=4200, verbose=0, class_weight=class_weights)
#     kfold = KFold(n_splits=3, shuffle=True, random_state=123)
#     score = cross_val_score(nn, X_train, y_train, scoring='f1_macro', cv=kfold, fit_params={'callbacks':[es]})
#     score = np.nan_to_num(score)
#     score = np.max(score) 
#     return score

# params_nn2 ={
# 'class1': (0.5, 3.0),
# 'class2': (0.3, 1.0)
# }

# nn_bo = BayesianOptimization(nn_cl_bo2, params_nn2, random_state=42)
# nn_bo.maximize(init_points=25, n_iter=4)
# best_params = nn_bo.max['params']
# best_accuracy = nn_bo.max['target']

# print(f"Best Hyperparameters: {best_params}")
# print(f"Best Cross-Validation Accuracy: {best_accuracy}")

In [519]:
# def nn_cl_bo1(class1, class2):
#     class_weights = [{0:class1, 1:class2}, {0:class2, 1:class1}]
    
#     es = EarlyStopping(monitor='val_loss', mode='min', verbose=0, patience=10)
#     nn = KerasClassifier(model=create_model(242), epochs=15, batch_size=4200, verbose=0, class_weight=class_weights)
#     kfold = KFold(n_splits=3, shuffle=True, random_state=123)
#     score = cross_val_score(nn, X_train1, y_train1, scoring='f1_macro', cv=kfold, fit_params={'callbacks':[es]})
#     score = np.nan_to_num(score)
#     score = np.max(score) 
#     return score

# params_nn1 = {
#     'class1': (5, 10.0),
#     'class2': (0.5, 1.0)
# }

# nn_bo1 = BayesianOptimization(nn_cl_bo1, params_nn1, random_state=42)
# nn_bo1.maximize(init_points=25, n_iter=4)
# best_params1 = nn_bo1.max['params']
# best_accuracy1 = nn_bo1.max['target']
# print(f"Best Hyperparameters: {best_params1}")
# print(f"Best Cross-Validation Accuracy: {best_accuracy1}")

### Обучение модели

In [517]:
model = create_model(X_train.shape[1], 142)

class_weights = {
    0: 1.35, # 1.31
    1: 0.51  # 0.53
}

history = model.fit(X_train, y_train,
                    epochs=15,
                    validation_data=(X_test, y_test),
                    batch_size=1000,
                    class_weight=class_weights,
                    verbose=2
                    )

Epoch 1/15
369/369 - 7s - 18ms/step - f1_score: 0.6009 - loss: 0.4007 - val_f1_score: 0.6179 - val_loss: 0.5186
Epoch 2/15
369/369 - 4s - 12ms/step - f1_score: 0.6163 - loss: 0.3914 - val_f1_score: 0.6209 - val_loss: 0.5073
Epoch 3/15
369/369 - 4s - 12ms/step - f1_score: 0.6180 - loss: 0.3899 - val_f1_score: 0.6219 - val_loss: 0.5085
Epoch 4/15
369/369 - 4s - 12ms/step - f1_score: 0.6210 - loss: 0.3887 - val_f1_score: 0.6231 - val_loss: 0.5090
Epoch 5/15
369/369 - 4s - 12ms/step - f1_score: 0.6220 - loss: 0.3878 - val_f1_score: 0.6242 - val_loss: 0.5002
Epoch 6/15
369/369 - 4s - 12ms/step - f1_score: 0.6229 - loss: 0.3872 - val_f1_score: 0.6249 - val_loss: 0.5003
Epoch 7/15
369/369 - 4s - 12ms/step - f1_score: 0.6238 - loss: 0.3867 - val_f1_score: 0.6253 - val_loss: 0.5023
Epoch 8/15
369/369 - 4s - 12ms/step - f1_score: 0.6238 - loss: 0.3865 - val_f1_score: 0.6259 - val_loss: 0.5014
Epoch 9/15
369/369 - 4s - 12ms/step - f1_score: 0.6242 - loss: 0.3861 - val_f1_score: 0.6267 - val_loss:

In [520]:
model2 = create_model(X_train1.shape[1], 242)

class_weights1 = {
    0: 8.3, # 8.3
    1: 0.69
}

history2 = model2.fit(X_train1, y_train1,
                    epochs=10,
                    validation_data=(X_test1, y_test1),
                    batch_size=1000,
                    class_weight=class_weights1,
                    verbose=2
                    )

Epoch 1/10
373/373 - 6s - 17ms/step - f1_score: 0.5491 - loss: 0.2491 - val_f1_score: 0.5670 - val_loss: 0.1410
Epoch 2/10
373/373 - 4s - 12ms/step - f1_score: 0.5666 - loss: 0.2321 - val_f1_score: 0.5691 - val_loss: 0.1382
Epoch 3/10
373/373 - 4s - 12ms/step - f1_score: 0.5721 - loss: 0.2280 - val_f1_score: 0.5672 - val_loss: 0.1414
Epoch 4/10
373/373 - 4s - 12ms/step - f1_score: 0.5723 - loss: 0.2258 - val_f1_score: 0.5708 - val_loss: 0.1334
Epoch 5/10
373/373 - 4s - 12ms/step - f1_score: 0.5724 - loss: 0.2240 - val_f1_score: 0.5681 - val_loss: 0.1350
Epoch 6/10
373/373 - 4s - 11ms/step - f1_score: 0.5735 - loss: 0.2227 - val_f1_score: 0.5709 - val_loss: 0.1308
Epoch 7/10
373/373 - 4s - 11ms/step - f1_score: 0.5737 - loss: 0.2214 - val_f1_score: 0.5726 - val_loss: 0.1349
Epoch 8/10
373/373 - 4s - 11ms/step - f1_score: 0.5751 - loss: 0.2208 - val_f1_score: 0.5676 - val_loss: 0.1362
Epoch 9/10
373/373 - 4s - 11ms/step - f1_score: 0.5764 - loss: 0.2198 - val_f1_score: 0.5695 - val_loss:

## Предсказание

In [521]:
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)

X_testFinal = X_testFinal.reindex(df_all.columns, axis=1, fill_value=0)
X_testFinal_data = scaler.transform(X_testFinal)
X_testFinal.describe()

,local_authority_highway_E060,local_authority_highway_E080,local_authority_highway_E090,local_authority_highway_E100,local_authority_highway_EHEA,local_authority_highway_S120,local_authority_highway_W060,casualty_class,sex_of_casualty,age_band_of_casualty,pedestrian_location,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_home_area_type,casualty_distance_banding,vehicle_type,vehicle_manoeuvre,engine_capacity_cc,age_of_vehicle,police_force,first_road_class,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.00000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.470322,1.366873,6.305587,0.745834,0.232699,0.051133,0.023673,7.616548,1.068289,1.587892,9.234791,19.306639,1242.074330,6.18414,28.323892,4.149598,5.238975,37.547730,3.767667,3.008446,1.095220,2.051848,1.634257,1.371369
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.722688,0.532466,2.455111,2.123633,0.626563,0.438217,0.223838,10.439010,0.917274,1.424338,11.294774,21.728723,1317.162953,6.47909,24.336775,1.469986,1.665436,14.710353,11.917428,2.759564,2.362987,1.732798,1.787258,0.933177
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.00000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,8.000000,9.000000,108.000000,0.00000,5.000000,3.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,1.000000,6.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,1.000000,9.000000,18.000000,1299.000000,5.00000,23.000000,4.000000,6.000000,30.000000,2.000000,3.000000,0.000000,1.000000,1.000000,1.000000
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.000000,2.000000,8.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,2.000000,9.000000,18.000000,1796.000000,11.00000,45.000000,6.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.000000,2.000000
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.000000,9.000000,11.000000,10.000000,9.000000,9.000000,2.000000,98.000000,3.000000,5.000000,99.000000,99.000000,16400.000000,97.00000,99.000000,6.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.000000,9.000000


In [522]:
import random

y1_final = model.predict(X_testFinal_data)
y2_final = model2.predict(X_testFinal_data)
tmp1 = []
tmp2 = []
for i in range(len(y1_final)):
    tmp1.append(np.argmax(y1_final[i])+2)
    tmp2.append(np.argmax(y2_final[i])+1)

y_final = [random.randint(1, 3) for i in range(X_testFinal_data.shape[0])]
for i in range(X_testFinal_data.shape[0]):
    if tmp2[i] == 1:
        y_final[i] = 1
    else: 
        y_final[i] = tmp1[i]

print(np.unique(y_final))

5199/5199 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step
5199/5199 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step
[1 2 3]


In [523]:
my_file = open("../predicts/nn_balance15.csv", "w+")

my_file.write("Id,casualty_severity\n")
my_i = 0
for i, row in X_testFinal.iterrows():
    my_file.write(f"{i}, {y_final[my_i]}\n")
    my_i = my_i+1
my_file.close()